# Delivery Delay Prediction with SAP HANA ML

This notebook demonstrates how to use SAP HANA Machine Learning (PAL) Unified Classification with a **Random Forest Classifier** on a Supply Chain Purchase Order dataset stored in SAP HANA Cloud.

| Item | Detail |
|------|--------|
| Training table | `SPURCHASE.PO_HISTORY` (600 records) |
| Prediction table | `SPURCHASE.PO_PREDICTION` (1 record) |
| Target | Last column of the dataset |
| Algorithm | Unified Classification – Random Forest |

---
## Step 1 – Install / Import Required Libraries

In [ ]:
%pip install hdbcli --break-system-packages
%pip install hana-ml --break-system-packages
%pip install folium --break-system-packages
%pip install ipywidgets --break-system-packages

%pip install plotly --break-system-packages
%pip install nbformat --break-system-packages
%pip install matplotlib --break-system-packages
%pip install -U python-dotenv

# kernel restart required!!!

In [ ]:
# Uncomment the line below if hana_ml is not yet installed
# !pip install hana_ml
import os
import hana_ml
from hana_ml import dataframe as hd
from hana_ml.algorithms.pal import metrics
from hana_ml.algorithms.pal.unified_classification import UnifiedClassification, json2tab_for_reason_code
from hana_ml.visualizers.unified_report import UnifiedReport
from dotenv import load_dotenv

import logging

# Load environment variables
load_dotenv()

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


print('hana_ml version:', hana_ml.__version__)

---
## Step 2 – Connect to SAP HANA Cloud

A **User Store Key** (`myUserKey`) is used so that credentials are not hard-coded in the notebook.  
Run `hdbuserstore SET myUserKey <host>:<port> <user> <password>` in a terminal once to register the key.

In [ ]:
from hana_ml import dataframe as hdf


# Get HANA credentials
HANA_USER = os.getenv('HANA_VECTOR_USER')
HANA_PASS = os.getenv('HANA_VECTOR_PASS')
HANA_HOST = os.getenv('HANA_VECTOR_HOST')

# Establish connection
myconn = hdf.ConnectionContext(
    address=HANA_HOST,
    port=443,
    user=HANA_USER,
    password=HANA_PASS
)
    
    
print(f"Connected to SAP HANA db version {myconn.hana_version()} \nat {HANA_HOST}:443 as user {HANA_USER}")

---
## Step 3 – Create HANA DataFrames

Data remains in HANA Cloud; only metadata is transferred to the Python client.

In [ ]:
SCHEMA    = 'SPURCHASE'
TBL_TRAIN = 'PO_HISTORY'
TBL_TEST  = 'PO_PREDICTION'

df_train = myconn.table(TBL_TRAIN, schema=SCHEMA)
df_test  = myconn.table(TBL_TEST,  schema=SCHEMA)

print('Training rows  :', df_train.count())
print('Prediction rows:', df_test.count())
print('Columns        :', df_train.columns)

### Identify the key column and the target (last) column

In [ ]:
# The target is the LAST column of the training table
label_col = df_train.columns[-1]

# The key is the FIRST column (or whichever column uniquely identifies a row)
key_col   = df_train.columns[0]

# All remaining columns are features
feature_cols = [c for c in df_train.columns if c not in (key_col, label_col)]

print(f'Key column   : {key_col}')
print(f'Label column : {label_col}')
print(f'Feature cols : {feature_cols}')

---
## Step 4 – Explore the Training Data

In [ ]:
# Preview first 5 rows
df_train.head(5).collect()

In [ ]:
# Statistical summary
df_train.describe().collect()

In [ ]:
# Distribution of the target class
df_train.agg([('count', label_col, 'COUNT')], group_by=label_col).collect()

In [ ]:
# Preview the prediction record
df_test.collect()

---
## Step 5 – Train the Model with Unified Classification (Random Forest)

In [ ]:
rdt_params = dict(random_state=2,
                  split_threshold=1e-7,
                  min_samples_leaf=1,
                  n_estimators=10,
                  max_depth=55)

uc_rfc = UnifiedClassification(
    func = 'RandomForest', **rdt_params
)

uc_rfc.fit(
    data        = df_train,
    key         = key_col,
    label       = label_col,
    features    = feature_cols,
    partition_method = 'stratified',   # stratified split for balanced classes
    stratified_column= 'Delivery_Delayed',
    partition_random_state = 42,
    training_percent = 0.8             # 80 % train / 20 % internal validation
)

print('Model training complete.')

### Visualize the model
In unifiedclassfication function, we provide a function generate_notebook_iframe_report() to visualize the results.

In [ ]:
from hana_ml.visualizers.dataset_report import DatasetReportBuilder

uc_rfc.build_report()

uc_rfc.generate_notebook_iframe_report()

---
## Step 6 – Evaluate the Model (Score)

> **Note:** Because the test table (`PO_PREDICTION`) contains only 1 record, scoring is performed on the internal validation split that was created during training.  
> If the test table contains the true label you can also call `uc_rfc.score(df_test, ...)` directly.

In [ ]:
# --- Option A: score on the training table's held-out validation split ---
# (uses the 20 % partition created by partition_method='stratified' above)
score_result = uc_rfc.score(
    data  = df_train,
    key   = key_col,
    label = label_col
)

print('Accuracy on validation split:', score_result)

### Confusion Matrix & Metrics

In [ ]:
# Detailed classification metrics stored in the model
uc_rfc.statistics_.collect()

In [ ]:
# Confusion matrix
uc_rfc.confusion_matrix_.collect()

In [ ]:
dtr_auc=uc_rfc.statistics_.filter("STAT_NAME='AUC'").cast('STAT_VALUE','DOUBLE').collect().at[0, 'STAT_VALUE']
dtr_auc

In [ ]:
uc_rfc.metrics_.collect()

In [ ]:
import matplotlib.pyplot as plt

tpr=uc_rfc.metrics_.filter("NAME='ROC_TPR'").select('Y').collect()
fpr=uc_rfc.metrics_.filter("NAME='ROC_FPR'").select('Y').collect()

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=1, label='ROC curve (area = %0.2f)' % dtr_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver operating characteristic')
plt.legend(loc="lower right")
plt.show()

---
## Step 7 – Predict on New Purchase Order Record

In [ ]:
predict_res = uc_rfc.predict(
    data = df_test,
    key  = key_col,
    features    = feature_cols
)

predict_res.collect()

In [ ]:
from hana_ml.visualizers.model_debriefing import TreeModelDebriefing

In [ ]:
shapley_explainer = TreeModelDebriefing.shapley_explainer(predict_res, df_test, key=key_col, label=label_col)
shapley_explainer.summary_plot()

---
## Step 8 – Feature Importance

In [ ]:
# Variable importance (Gini / Mean Decrease in Impurity)
importance = uc_rfc.importance_
importance.sort('IMPORTANCE', desc=True).collect()

---
## Step 9 – Model Report (Unified Report)

In [ ]:
UnifiedReport(uc_rfc).build().display()

---
## Step 10 – Close Connection

In [ ]:
myconn.close()
print('Connection closed.')